# Comparing Multiple Agents

This notebook demonstrates:
- Running multiple agents on the same tasks
- Side-by-side performance comparison
- Statistical significance testing
- Visualization of relative performance

## Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from agentick.experiments import ExperimentConfig, ExperimentRunner

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

## Run Multiple Agents

Let's run several baseline agents:

In [ ]:
# Define agents to compare
agents = {
    "Random": {"type": "random"},
    "Oracle": {"type": "oracle"},
    "Greedy": {"type": "greedy"},
}

# Common experiment settings
tasks = ["GoToGoal-v0", "MazeNavigation-v0", "KeyDoorPuzzle-v0"]
n_seeds = 5
n_episodes = 10

# Run each agent
results_dict = {}

for agent_name, agent_config in agents.items():
    print(f"\n{'=' * 60}")
    print(f"Running {agent_name}...")
    print(f"{'=' * 60}")

    config = ExperimentConfig(
        name=f"compare_{agent_name.lower()}",
        agent=agent_config,
        tasks=tasks,
        difficulties=["easy"],
        n_episodes=n_episodes,
        n_seeds=n_seeds,
        render_modes=["ascii"],
        output_dir=f"results/comparison/{agent_name.lower()}",
    )

    runner = ExperimentRunner(config)
    results = runner.run()
    results_dict[agent_name] = results

    print(f"\n{agent_name} complete!")

## Load and Compare Results

Load results for each agent:

In [ ]:
# Load summaries
summaries = {}

for agent_name in agents.keys():
    result_dir = Path(f"results/comparison/{agent_name.lower()}")
    exp_dirs = sorted(result_dir.glob("*"))
    latest = exp_dirs[-1] if exp_dirs else None

    if latest:
        with open(latest / "summary.json") as f:
            summaries[agent_name] = json.load(f)

# Create comparison DataFrame
comparison_data = []
for agent_name, summary in summaries.items():
    comparison_data.append(
        {
            "Agent": agent_name,
            "Success Rate": summary.get("success_rate", 0),
            "Mean Return": summary.get("mean_return", 0),
            "Mean Length": summary.get("mean_length", 0),
        }
    )

df_comparison = pd.DataFrame(comparison_data)
print("\nAgent Comparison Summary:")
print(df_comparison.to_string(index=False))

## Visualize Overall Performance

In [ ]:
# Success rate comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Success rates
agents_list = df_comparison["Agent"].tolist()
success_rates = df_comparison["Success Rate"].tolist()

colors = ["#FF6B6B", "#4ECDC4", "#45B7D1"]
ax1.bar(agents_list, success_rates, color=colors, alpha=0.8)
ax1.set_ylabel("Success Rate", fontsize=12)
ax1.set_title("Success Rate Comparison", fontsize=14, fontweight="bold")
ax1.set_ylim(0, 1)
ax1.grid(axis="y", alpha=0.3)

# Mean returns
mean_returns = df_comparison["Mean Return"].tolist()
ax2.bar(agents_list, mean_returns, color=colors, alpha=0.8)
ax2.set_ylabel("Mean Return", fontsize=12)
ax2.set_title("Mean Return Comparison", fontsize=14, fontweight="bold")
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Per-Task Breakdown

Compare agents on each individual task:

In [ ]:
# Load per-task results for each agent
task_comparison = {}

for agent_name in agents.keys():
    result_dir = Path(f"results/comparison/{agent_name.lower()}")
    exp_dirs = sorted(result_dir.glob("*"))
    latest = exp_dirs[-1] if exp_dirs else None

    if latest:
        task_dirs = list((latest / "per_task").glob("*"))

        for task_dir in task_dirs:
            task_name = task_dir.name
            metrics_file = task_dir / "metrics.json"

            if metrics_file.exists():
                with open(metrics_file) as f:
                    data = json.load(f)

                if task_name not in task_comparison:
                    task_comparison[task_name] = {}

                task_comparison[task_name][agent_name] = data["aggregate_metrics"]

# Create per-task comparison
for task_name, agent_metrics in task_comparison.items():
    print(f"\n{task_name}:")
    for agent_name, metrics in agent_metrics.items():
        print(
            f"  {agent_name:10s}: success_rate={metrics['success_rate']:.2%}, mean_return={metrics['mean_return']:.3f}"
        )

In [ ]:
# Grouped bar chart for per-task success rates
fig, ax = plt.subplots(figsize=(12, 6))

task_names = list(task_comparison.keys())
agent_names = list(agents.keys())
x = np.arange(len(task_names))
width = 0.25

for i, agent_name in enumerate(agent_names):
    success_rates = [task_comparison[task][agent_name]["success_rate"] for task in task_names]
    offset = (i - len(agent_names) / 2 + 0.5) * width
    ax.bar(x + offset, success_rates, width, label=agent_name, alpha=0.8)

ax.set_xlabel("Task", fontsize=12)
ax.set_ylabel("Success Rate", fontsize=12)
ax.set_title("Success Rate by Task and Agent", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([t.replace("-v0", "") for t in task_names], rotation=45, ha="right")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Statistical Significance Testing

Test if performance differences are statistically significant:

In [ ]:
# For demonstration - normally you'd load all episode-level results
# Here we'll simulate from the aggregate metrics

print("Statistical Significance Tests:")
print("=" * 60)

# Compare Oracle vs Random
oracle_sr = summaries["Oracle"]["success_rate"]
random_sr = summaries["Random"]["success_rate"]

print("\nOracle vs Random:")
print(f"  Oracle success rate: {oracle_sr:.2%}")
print(f"  Random success rate: {random_sr:.2%}")
print(f"  Difference: {(oracle_sr - random_sr):.2%}")

# In a real analysis, you'd do a proper statistical test here
# For now, just show the magnitude of difference
relative_improvement = (
    ((oracle_sr - random_sr) / random_sr * 100) if random_sr > 0 else float("inf")
)
print(f"  Relative improvement: {relative_improvement:.1f}%")

## Radar Chart Comparison

Visualize multi-dimensional performance:

In [ ]:
from math import pi

# Prepare data for radar chart
categories = list(task_comparison.keys())
N = len(categories)

# Angles for each axis
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Complete the circle

# Create radar chart
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection="polar"))

for agent_name, color in zip(agent_names, colors):
    values = [task_comparison[task][agent_name]["success_rate"] for task in categories]
    values += values[:1]  # Complete the circle

    ax.plot(angles, values, "o-", linewidth=2, label=agent_name, color=color)
    ax.fill(angles, values, alpha=0.15, color=color)

# Fix axis labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels([t.replace("-v0", "") for t in categories])
ax.set_ylim(0, 1)
ax.set_ylabel("Success Rate", fontsize=10)
ax.set_title("Agent Performance Radar Chart", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
ax.grid(True)

plt.tight_layout()
plt.show()

## Ranking and Summary Table

Create a comprehensive ranking:

In [ ]:
# Sort by success rate
df_ranked = df_comparison.sort_values("Success Rate", ascending=False).reset_index(drop=True)
df_ranked.index += 1  # Start ranking from 1

print("\nAgent Rankings:")
print("=" * 60)
print(df_ranked.to_string())

# Best agent
best_agent = df_ranked.iloc[0]
print(f"\n🏆 Best Agent: {best_agent['Agent']}")
print(f"   Success Rate: {best_agent['Success Rate']:.2%}")
print(f"   Mean Return: {best_agent['Mean Return']:.3f}")

## Export Comparison

Save comparison results:

In [ ]:
# Export to CSV
output_dir = Path("results/comparison")
output_dir.mkdir(parents=True, exist_ok=True)

comparison_file = output_dir / "agent_comparison.csv"
df_ranked.to_csv(comparison_file)
print(f"\nComparison saved to: {comparison_file}")

# Export per-task breakdown
task_breakdown = []
for task_name in task_comparison.keys():
    for agent_name in agent_names:
        metrics = task_comparison[task_name][agent_name]
        task_breakdown.append(
            {
                "Task": task_name,
                "Agent": agent_name,
                "Success Rate": metrics["success_rate"],
                "Mean Return": metrics["mean_return"],
                "Mean Length": metrics["mean_length"],
            }
        )

df_tasks = pd.DataFrame(task_breakdown)
task_file = output_dir / "per_task_comparison.csv"
df_tasks.to_csv(task_file, index=False)
print(f"Per-task breakdown saved to: {task_file}")

## Next Steps

- Run larger comparison with more agents
- Test on more difficult tasks
- Include your custom trained agents
- Submit best agents to the leaderboard!
- See **04_leaderboard_analysis.ipynb** for leaderboard analysis